Steam Comment Scraping: https://partner.steamgames.com/doc/store/getreviews

Below is a work in progress. There appears to be a hiccup when I run it on a larger date range

In [ ]:
# API Best Practices:
# minimize API calls: put them in separate cells from other code so we only need to run it once
# respect rate limits: check API docs to see if they're stated (we can get blocked lol)
# check the content: APIs can send "bad" data with a 200
# set headers: we want to id ourselfs seperate from general colab traffic

Next the code chunks below will be creating functions that allow us to scrape the APIs

In [ ]:
os.mkdir(path = "../data")
os.mkdir(path = '../data/download')

FileExistsError: [Errno 17] File exists: '../data'

In [ ]:
# Importing necessary libraries

import csv
import datetime as date
import json
import os
import time
import statistics

import numpy as np
import pandas as pd
import requests

from requests.exceptions import SSLError

In [ ]:
# Creating function that will request information from API

def request(url, parameters = None):

  try:
      headers = {'User-Agent': 'Dat490SteamAnalysis'}
      response = requests.get(url = url, params = parameters
                              , headers = headers)

      if not response.status_code == 200:
        print("Error", response.status_code)
      #print("response data: ", response.content)

  except SSLError as s:
      print("SSL Error: ", s)

      for i in range(5, 0, -1):
          print('\rWaiting... ({})'.format(i), end = '')
          time.sleep(1)
      print('\rRetrying' + ' ' * 10)

      # Recursively try the function again
      return request(url, parameters)

  if response:
    #print("response data: ", response.content)
    return response.json()
  else:
    print('No response, waiting 60 seconds...')
    time.sleep(60)
    print('Retrying')
    return request(url, parameters)

In [ ]:
# Scraping SteamSpy for game ids
i = 0
dicts = {}
is_last_page = False

while not is_last_page:
  url = 'https://steamspy.com/api.php'
  params = {'request':'all','page':i}
  jsdata = request(url, parameters = params)
  dicts.update(jsdata)

  if len(jsdata) < 1000:
    print('less than 1000 entries on this page')
    print('page', i)
    is_last_page = True

  else:
    i += 1
    time.sleep(61)
    print('page', i)

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ssall = pd.DataFrame.from_dict(dicts, orient = 'index')

id_list = ssall[['appid', 'name']].sort_values('appid').reset_index(drop = True)

print(id_list.head())
print(id_list.shape)

   appid                       name
0     10             Counter-Strike
1     20      Team Fortress Classic
2     30              Day of Defeat
3     40         Deathmatch Classic
4     50  Half-Life: Opposing Force
(73298, 2)


In [ ]:
verify = pd.read_csv('/content/drive/My Drive/id_list.csv')
print(verify.head())
print(verify.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/id_list.csv'

In [ ]:
# These next functions will allow us to get game data
# Due to the mass amount of games, we will also create a function
# that allows us to batch the amount of data we request

def get_gamedata(start, stop, parser, pause, id_list):

  game_data = []

  for index, row in id_list[start:stop].iterrows():
    print('Current index: {}'.format(index), end = '\r')

    gameid = row['appid']
    name = row['name']

    data = parser(gameid, name)
    game_data.append(data)

    time.sleep(pause)

  return game_data

def process_batches(file_num, parser, idx_path, download_path, columns, begin = 0, end = -1, batchsize = 100, pause = 1):


  id_list_filename = f'id_list{file_num}.csv'
  id_list = pd.read_csv(idx_path+id_list_filename)

  if end == -1:
    end = len(id_list)

  #pd.read_csv(os.path.join(download_path, id_list_filename))

  # setting output filenames based on file_num:
  data_name = f'game_data{file_num}.csv'
  index_name = f'index{file_num}.txt'

  print(f'Starting at index {begin} for file_num {file_num}: \n')

  if end == -1:
    end = len(id_list) + 1

  batches = np.arange(begin, end, batchsize)
  batches = np.append(batches, end)

  games_written = 0
  batch_time = []

  for i in range(len(batches) - 1):
    start_time = time.time()

    start = batches[i]
    stop = batches[i + 1]

    game_data = get_gamedata(start, stop, parser, pause, id_list)

    rel_path = os.path.join(download_path, data_name)

    with open(rel_path, 'a', newline = '', encoding = 'utf-8') as f:
      writer = csv.DictWriter(f, fieldnames = columns, extrasaction = 'ignore')

      for j in range(3, 0, -1):
        print("\rAbout to write data, don't stop script! ({})".format(j), end = '')
        time.sleep(0.5)

      writer.writerows(game_data)
      print('\rExported lines {}-{} to {}'.format(start, stop-1, data_name), end = ' ')

    games_written += len(game_data)

    with open(download_path+index_name, 'w') as f:
      index = stop
      print(index, file = f)

    end_time = time.time()
    time_taken = end_time - start_time

    batch_time.append(time_taken)
    mean_time = statistics.mean(batch_time)

    est_remaining = (len(batches) - i - 2) * mean_time

    remaining_td = date.timedelta(seconds = round(est_remaining))
    time_td = date.timedelta(seconds = round(time_taken))
    mean_td = date.timedelta(seconds = round(mean_time))

    print('Batch {} time: {} (avg: {}, remaining: {})'.format(i, time_td, mean_td, remaining_td))

  print('\nProcessing batches complete. {} games written!'.format(games_written))

In [ ]:
def reset_idx(download_path, file_num):
  index_name = f'index{file_num}.txt'

  rel_path = os.path.join(download_path, index_name)

  with open('/content/drive/My Drive/Dat490/Data/'+ index_name, 'w') as f:
    print(0, file = f)

def get_idx(download_path, file_num):
    index_name = f'index{file_num}.txt'
    rel_path = os.path.join(download_path, index_name)
    try:
        with open('/content/drive/My Drive/Dat490/Data/'+ index_name, 'r') as f:
            idx = int(f.readline().strip())
            print('Current index:', idx)
    except FileNotFoundError:
        idx = 0
        with open('/content/drive/My Drive/Dat490/Data/'+ index_name, 'w') as f:
          f.write('0')
        print('Index file not found, starting from index 0.')

    return idx

def prepare_data(download_path, file_num, idx, columns):
  filename = f'game_data{file_num}.csv'

  if idx == 0:
    rel_path = os.path.join(download_path, filename)

    with open(rel_path, 'w', newline = '') as f:
      writer = csv.DictWriter(f, fieldnames = columns)
      writer.writeheader()

In [ ]:
import os
#In this chunk we will be downloading the Steam Data

def parse_steam(gameid, name):
    url = f'http://store.steampowered.com/api/appdetails?appids={gameid}'
    params = {'gameid': gameid}

    jsdata = request(url, parameters=params)
    jsgamedata = jsdata[str(gameid)]

    if jsgamedata['success']:
        data = jsgamedata['data']
        if 'release_date' in data:
            data['release_date'] = data['release_date']['date']  # Accessing the 'date' field inside 'release_date'
        else:
            data['release_date'] = 'Unknown'  # Setting a default if no release date is found
    else:
        data = {'name': name, 'steam_gameid': gameid, 'release_date': 'Unknown'}

    return data

download_path = '/content/drive/My Drive/Dat490/Data/'
idx_path = '/content/drive/My Drive/Dat490/'
steam_columns = ['type', 'name', 'steam_appid', 'required_age', 'is_free', 'controller_support',
                 'dlc', 'detailed_description', 'about_the_game', 'short_description', 'fullgame',
                 'supported_languages', 'header_image', 'website', 'pc_requirements', 'mac_requirements',
                 'linux_requirements', 'legal_notice', 'drm_notice', 'ext_user_account_notice',
                 'developers', 'publishers', 'demos', 'price_overview', 'packages', 'package_groups',
                 'platforms', 'metacritic', 'reviews', 'categories', 'genres', 'screenshots',
                 'movies', 'recommendations', 'achievements', 'release_date', 'support_info',
                 'background', 'content_descriptors']


In [ ]:
file_num = 4

idx = get_idx(download_path, file_num)

if idx == 0:
  prepare_data(download_path, file_num, idx, steam_columns)

process_batches(file_num, parse_steam, idx_path, download_path, steam_columns, batchsize=20, pause=1.1, begin=idx, end = 16712)

Current index: 16060
Starting at index 16060 for file_num 4: 

Exported lines 16060-16079 to game_data4.csv Batch 0 time: 0:00:28 (avg: 0:00:28, remaining: 0:15:10)
Exported lines 16080-16099 to game_data4.csv Batch 1 time: 0:00:29 (avg: 0:00:29, remaining: 0:14:52)
Exported lines 16100-16119 to game_data4.csv Batch 2 time: 0:00:31 (avg: 0:00:29, remaining: 0:14:42)
Exported lines 16120-16139 to game_data4.csv Batch 3 time: 0:00:29 (avg: 0:00:29, remaining: 0:14:08)
Exported lines 16140-16159 to game_data4.csv Batch 4 time: 0:00:29 (avg: 0:00:29, remaining: 0:13:38)
Exported lines 16160-16179 to game_data4.csv Batch 5 time: 0:00:29 (avg: 0:00:29, remaining: 0:13:06)
Exported lines 16180-16199 to game_data4.csv Batch 6 time: 0:00:29 (avg: 0:00:29, remaining: 0:12:37)
Exported lines 16200-16219 to game_data4.csv Batch 7 time: 0:00:30 (avg: 0:00:29, remaining: 0:12:10)
Exported lines 16220-16239 to game_data4.csv Batch 8 time: 0:00:29 (avg: 0:00:29, remaining: 0:11:41)
Exported lines 1624

In [ ]:
idx = idx + 1

In [ ]:
!ls '/content/drive/My Drive/Dat490/Data/'

game_data1.csv	game_data3.csv	   index1.txt  index3.txt
game_data2.csv	game_data3.gsheet  index2.txt


In [ ]:
#reset_idx(download_path, file_num)

In [ ]:
#turning the list into 3 parts:
df.shape
size = 25000
df1 = df.iloc[:size]
df2 = df.iloc[size:size*2]
df3 = df.iloc[size*2:]

df1.to_csv(id_path + '1'+ end, index=False)
df2.to_csv(id_path + '2'+ end, index=False)
df3.to_csv(id_path + '3'+ end, index=False)

In [ ]:
# fixing 2:
df_ids4 = pd.read_csv('/content/drive/My Drive/Dat490/id_list4.csv')
df_ids2 = pd.read_csv('/content/drive/My Drive/Dat490/id_list2.csv')

# if id is in id4, remove from id2
df_ids_cleaned = df_ids2[~df_ids2['appid'].isin(df_ids4['appid'])]
df_ids_cleaned.to_csv('/content/drive/My Drive/Dat490/id_list2_cleaned.csv', index=False)